# PT03: Automatic differentiation from scratch: solutions

The solutions to the exercises of [PT03](https://colab.research.google.com/github/sscardapane/teaching-materials/blob/main/notebooks/PT03_Automatic_differentiation.ipynb). Each solution comes under the heading it has in the lab, with a short explanation followed by the code. If you fall behind during the lab, copy the solution cell into your notebook in place of your version and continue from there.

## 2. From the chain rule to VJPs

### The same idea for vectors and matrices

$\bar y \sim (3)$, $\bar W \sim (3, 4)$ and $\bar x \sim (4)$: each adjoint has the shape of its variable.
Since $\langle v, Ux\rangle = \sum_{i,j} v_i U_{ij} x_j = \langle v x^\top, U\rangle$,
we get $\bar W = \bar y\,x^\top$ (an outer product) and, in the same way,
$\bar x = W^\top \bar y$. Activity 1a uses the same formulas with a matrix $B$ in place of $x$.

## 4. Activity 1: local VJPs

### 1a. Addition, multiplication and matrix products ★★

The three functions return the formulas above. For the matrix product, `v` has the shape $(n, m)$ of the output: `v @ b.T` has shape $(n, k)$, like `a`, and `a.T @ v` has shape $(k, m)$, like `b`. Checking these shapes is a quick way to get the transposes right.

In [ ]:
def add_vjp(a, b, v):
    return v, v

def multiply_vjp(a, b, v):
    return v * b, v * a

def matmul_vjp(a, b, v):
    return v @ b.T, a.T @ v

### 1b. Nonlinear operations ★

Both functions multiply `v` elementwise by the derivative of the operation at each entry. For `tanh` the derivative is $1 - y^2$, computed from the saved output. For `relu`, `(x > 0)` is an array of booleans that NumPy treats as ones and zeros: it is zero where $x \le 0$, including $x = 0$.

In [ ]:
def tanh_vjp(y, v):
    return (1 - y ** 2) * v

def relu_vjp(x, v):
    return (x > 0) * v

## 5. Activity 2: `backward` ★

Going through the topological order in reverse handles every node after all the nodes that use it, so its adjoint is complete when we call its `vjp`. `adjoints.get(parent, 0.) + contribution` adds the contributions instead of overwriting them, so a parent used twice, as in `x * x`, receives both. A leaf adds its adjoint to `.grad`; the first time it stores a copy, so that a later accumulation does not change an array returned earlier.

In [ ]:
def backward(self, gradient=None):
    adjoints = {self: initial_adjoint(self, gradient)}
    for node in reversed(topological_order(self)):
        adjoint = adjoints[node]
        if node.vjp is None:  # A leaf: accumulate into .grad, as PyTorch does.
            node.grad = adjoint.copy() if node.grad is None else node.grad + adjoint
            continue
        for parent, contribution in zip(node.parents, node.vjp(adjoint)):
            if parent.requires_grad:
                adjoints[parent] = adjoints.get(parent, 0.) + contribution

Tensor.backward = backward

## 6. Activity 3: training the PT02 classifier ★

The parameters become new leaves at every call, so their `.grad` starts empty and there is nothing to clear: this replaces setting `.grad = None` in PT02. After `loss.backward()` the gradients are in `weight.grad` and `bias.grad`. The updated parameters are new arrays, computed outside the graph; the next call builds a new graph from them.

In [ ]:
def classifier_step(X, targets, W, b, lr):
    weight = Tensor(W, requires_grad=True, name='W')
    bias = Tensor(b, requires_grad=True, name='b')
    logits = Tensor(X, name='X') @ weight.T + bias
    loss = cross_entropy(logits, targets)
    loss.backward()
    dw, db = weight.grad, bias.grad
    return dict(logits=logits.data.copy(), loss=loss.data.item(),
                dW=dw, db=db, W=W - lr * dw, b=b - lr * db)

## 8. What you built

1. Each $b_c$ is reused for all $B$ examples, so its adjoint is the sum of $B$
   contributions. $W$ enters through a matrix product, whose VJP
   $\bar W = \bar Z^\top X$ already sums over the batch.
2. Node identity lets backward compute $\bar u$ once and propagate it once.
   The two edges carry two contributions that must be added; overwriting gives,
   for example, $x$ instead of $2x$ as the derivative of `(x * x)`.
3. A node handled before all its consumers have contributed propagates an
   incomplete adjoint (12 instead of 44 in Section 5). Correct local rules are
   not enough; the order matters too.
4. $\tanh'(x) = 1 - \tanh(x)^2$ is cheap to compute from the output, which the
   forward pass already produced. Saving the input would also work but would
   require computing $\tanh$ again.

## Optional A: an MLP with your engine ★★

As in `classifier_step`, the parameters become new leaves at every call. The forward pass uses only operations that `Tensor` already has: two matrix products, two broadcast additions and `relu`. One `backward()` fills the four `.grad` fields.

In [ ]:
def mlp_step(X, targets, params, lr):
    leaves = {name: Tensor(value, requires_grad=True, name=name) for name, value in params.items()}
    hidden = (Tensor(X, name='X') @ leaves['W1'].T + leaves['b1']).relu()
    loss = cross_entropy(hidden @ leaves['W2'].T + leaves['b2'], targets)
    loss.backward()
    return loss.data.item(), {name: params[name] - lr * leaves[name].grad for name in params}

## Optional B: gradients as functions ★

`grad` returns a new function, `evaluate`. Each call wraps the input in a new leaf, runs `function` and calls `backward()`. If the output does not require gradients, it does not depend on `x`, so the gradient is zero.

In [ ]:
def grad(function):
    def evaluate(value):
        x = Tensor(value, requires_grad=True, name='input')
        output = as_tensor(function(x))
        if output.data.shape != ():
            raise ValueError('grad requires a scalar output')
        if not output.requires_grad:  # The output does not depend on x.
            return np.zeros_like(x.data)
        output.backward()
        return x.grad
    return evaluate

## Optional C: a Jacobian from VJPs ★★

One forward pass builds the graph. Each call to `backward` with the one-hot seed $e_j$ returns row $j$ of the Jacobian; `x.grad` is reset before each call because `backward` accumulates. The final reshape gives the shape `output.shape + input.shape`.

A function from $\mathbb R^{1000}$ to $\mathbb R$ needs one backward pass, and one from $\mathbb R$ to $\mathbb R^{1000}$ needs 1000. Forward mode needs one JVP per input entry, so a single pass for the second function. Training needs only the first case: one scalar loss, one seed.

In [ ]:
def jacobian(function, value):
    x = Tensor(value, requires_grad=True)
    output = as_tensor(function(x))
    result = np.zeros((output.data.size,) + x.data.shape)
    if output.requires_grad:  # Otherwise the output does not depend on x.
        for row in range(output.data.size):
            seed = np.zeros_like(output.data)
            seed.reshape(-1)[row] = 1
            x.grad = None
            output.backward(seed)
            result[row] = x.grad
    return result.reshape(output.data.shape + x.data.shape)